<a href="https://www.kaggle.com/code/jagrutiyuvrajdhangar/marginal-competition?scriptVersionId=299309166" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/march-machine-learning-mania-2026/Conferences.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneyDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneySeedRoundSlots.csv
/kaggle/input/march-machine-learning-mania-2026/MRegularSeasonDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/MGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/WSecondaryTourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2026/WGameCities.csv
/kaggle/input/march-machine-learning-mania-2026/MSeasons.csv
/kaggle/input/march-machine-learning-mania-2026/WNCAATourneySlots.csv
/kaggle/input/march-machine-learning-mania-2026/MSecondaryTourneyTeams.csv
/kaggle/input/march-machine-learning-mania-2026/SampleSubmissionStage2.csv
/kaggle/input/march-machine-learning-mania

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss
import re
import warnings
warnings.filterwarnings("ignore")

In [3]:
DATA_PATH = "/kaggle/input/march-machine-learning-mania-2026/"


regular = pd.read_csv(DATA_PATH + "MRegularSeasonDetailedResults.csv")
tourney = pd.read_csv(DATA_PATH + "MNCAATourneyDetailedResults.csv")
seeds = pd.read_csv(DATA_PATH + "MNCAATourneySeeds.csv")
sample = pd.read_csv(DATA_PATH + "SampleSubmissionStage2.csv")

In [4]:
def add_advanced_stats(df):
    
    # Winning team possessions
    df["WPoss"] = df["WFGA"] - df["WOR"] + df["WTO"] + 0.475 * df["WFTA"]
    df["LPoss"] = df["LFGA"] - df["LOR"] + df["LTO"] + 0.475 * df["LFTA"]
    
    # Offensive Efficiency
    df["WOffEff"] = df["WScore"] / df["WPoss"]
    df["LOffEff"] = df["LScore"] / df["LPoss"]
    
    # Defensive Efficiency
    df["WDefEff"] = df["LScore"] / df["LPoss"]
    df["LDefEff"] = df["WScore"] / df["WPoss"]
    
    # Net Rating
    df["WNet"] = df["WOffEff"] - df["WDefEff"]
    df["LNet"] = df["LOffEff"] - df["LDefEff"]
    
    return df

regular = add_advanced_stats(regular)

In [5]:
def build_team_stats(df):
    
    w_cols = ["Season", "WTeamID", "WScore", "WOffEff", "WDefEff", "WNet"]
    l_cols = ["Season", "LTeamID", "LScore", "LOffEff", "LDefEff", "LNet"]
    
    w_df = df[w_cols].copy()
    w_df.columns = ["Season", "TeamID", "Score", "OffEff", "DefEff", "Net"]
    w_df["Win"] = 1
    
    l_df = df[l_cols].copy()
    l_df.columns = ["Season", "TeamID", "Score", "OffEff", "DefEff", "Net"]
    l_df["Win"] = 0
    
    full = pd.concat([w_df, l_df])
    
    team_stats = full.groupby(["Season", "TeamID"]).agg({
        "Score": "mean",
        "OffEff": "mean",
        "DefEff": "mean",
        "Net": "mean",
        "Win": "mean"
    }).reset_index()
    
    return team_stats

team_stats = build_team_stats(regular)

In [6]:
def seed_to_int(seed):
    return int(re.sub("[^0-9]", "", seed))

seeds["SeedNum"] = seeds["Seed"].apply(seed_to_int)

In [7]:
def build_training_data(tourney, team_stats, seeds):

    games = tourney[["Season", "WTeamID", "LTeamID"]].copy()

    # Create positive examples
    df1 = games.copy()
    df1["Team1"] = df1["WTeamID"]
    df1["Team2"] = df1["LTeamID"]
    df1["Target"] = 1

    # Create negative examples (reverse teams)
    df2 = games.copy()
    df2["Team1"] = df2["LTeamID"]
    df2["Team2"] = df2["WTeamID"]
    df2["Target"] = 0

    df = pd.concat([df1, df2], ignore_index=True)

    # Merge team stats
    df = df.merge(team_stats, left_on=["Season", "Team1"],
                  right_on=["Season", "TeamID"])
    df = df.merge(team_stats, left_on=["Season", "Team2"],
                  right_on=["Season", "TeamID"],
                  suffixes=("_1", "_2"))

    # Merge seeds
    df = df.merge(seeds[["Season","TeamID","SeedNum"]],
                  left_on=["Season","Team1"],
                  right_on=["Season","TeamID"])
    df.rename(columns={"SeedNum":"Seed1"}, inplace=True)

    df = df.merge(seeds[["Season","TeamID","SeedNum"]],
                  left_on=["Season","Team2"],
                  right_on=["Season","TeamID"])
    df.rename(columns={"SeedNum":"Seed2"}, inplace=True)

    # Feature Differences
    df["NetDiff"] = df["Net_1"] - df["Net_2"]
    df["OffDiff"] = df["OffEff_1"] - df["OffEff_2"]
    df["DefDiff"] = df["DefEff_1"] - df["DefEff_2"]
    df["WinDiff"] = df["Win_1"] - df["Win_2"]
    df["SeedDiff"] = df["Seed1"] - df["Seed2"]

    features = ["NetDiff", "OffDiff", "DefDiff", "WinDiff", "SeedDiff"]

    return df[["Season"] + features + ["Target"]]
    
    return df[["Season"] + features + ["Target"]]

train_df = build_training_data(tourney, team_stats, seeds)

In [8]:
train_df = build_training_data(tourney, team_stats, seeds)

print(train_df["Target"].value_counts())

Target
1    1449
0    1449
Name: count, dtype: int64


In [9]:
features = ["NetDiff", "OffDiff", "DefDiff", "WinDiff", "SeedDiff"]

X = train_df[features]
y = train_df["Target"]
groups = train_df["Season"]

gkf = GroupKFold(n_splits=5)

models = []
scores = []

for train_idx, val_idx in gkf.split(X, y, groups):
    
    model = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.01,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8
    )
    
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    
    preds = model.predict_proba(X.iloc[val_idx])[:,1]
    score = log_loss(y.iloc[val_idx], preds)
    
    scores.append(score)
    models.append(model)

print("CV LogLoss:", np.mean(scores))

[LightGBM] [Info] Number of positive: 1184, number of negative: 1184
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000696 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1049
[LightGBM] [Info] Number of data points in the train set: 2368, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 1184, number of negative: 1184
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000136 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1049
[LightGBM] [Info] Number of data points in the train set: 2368, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 1184, number of negative: 1184
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the

In [10]:
def build_submission(sample, team_stats, seeds, models):
    
    preds = []
    
    for _, row in sample.iterrows():
        season, t1, t2 = row["ID"].split("_")
        season = int(season)
        t1 = int(t1)
        t2 = int(t2)
        
        s1 = team_stats[(team_stats.Season==season) & (team_stats.TeamID==t1)]
        s2 = team_stats[(team_stats.Season==season) & (team_stats.TeamID==t2)]
        
        if len(s1)==0 or len(s2)==0:
            preds.append(0.5)
            continue
        
        s1 = s1.iloc[0]
        s2 = s2.iloc[0]
        
        seed1 = seeds[(seeds.Season==season)&(seeds.TeamID==t1)]["SeedNum"].values
        seed2 = seeds[(seeds.Season==season)&(seeds.TeamID==t2)]["SeedNum"].values
        
        if len(seed1)==0 or len(seed2)==0:
            preds.append(0.5)
            continue
        
        features = np.array([[
            s1["Net"] - s2["Net"],
            s1["OffEff"] - s2["OffEff"],
            s1["DefEff"] - s2["DefEff"],
            s1["Win"] - s2["Win"],
            seed1[0] - seed2[0]
        ]])
        
        prob = np.mean([m.predict_proba(features)[:,1][0] for m in models])
        preds.append(prob)
    
    sample["Pred"] = preds
    return sample

submission = build_submission(sample, team_stats, seeds, models)
submission.to_csv("submission.csv", index=False)

print("Submission file created!")

Submission file created!
